# EVSC 410 — Morphodynamic laboratory 1

**Question:** How does changing upstream sediment supply alter a gravel-bed river profile?

This is a simplified **uniform-width**, single-grain-size, normal-flow teaching model. It is not a restoration design tool.

**Before running:** predict whether sediment deficit, equilibrium feed, and sediment surplus will cause aggradation or degradation. Explain your prediction using Exner sediment continuity.

In [ ]:
from dataclasses import replace
import numpy as np
import matplotlib.pyplot as plt
from teaching_model import Parameters, classroom_estimate, simulate

p = Parameters(length_m=300, dx_m=5, width_m=8, initial_slope=.008,
               discharge_m3s=8, grain_m=.025, duration_s=1800,
               max_dt_s=2, save_count=7)


## 1. Numerical preflight — required

Hydraulic Courant number: $C_h=U\Delta t/\Delta x$ (informational for quasi-steady normal flow).

Morphodynamic stability number: $C_m=D_m\Delta t/\Delta x^2$, where $D_m=(1-\lambda_p)^{-1}\partial q_s/\partial S$.

The code uses a conservative safety coefficient of 0.20 for its explicit Exner discretization and updates this restriction during a simulation. Initial diagnostics cannot guarantee that later bed slopes remain valid.

In [ ]:
d = classroom_estimate(p)
for name in ['cells','depth_m','velocity_ms','hydraulic_courant',
             'morphodynamic_number','max_morphodynamic_dt_s',
             'planned_dt_s','estimated_steps','stored_profiles_kib']:
    print(f'{name:32s} {d[name]:.5g}')
if d['caution']:
    print('CAUTION: large run for a browser. Reduce cells, duration, or saved profiles.')
else:
    print('Initial run-size estimate is suitable for classroom exploration.')


## 2. Run the experiment

Run three cases, changing **only** the sediment feed ratio. The model retains a few snapshots rather than all time steps.

In [ ]:
results = {}
for ratio in [0.7, 1.0, 1.3]:
    scenario = replace(p, feed_ratio=ratio)
    info = classroom_estimate(scenario)
    if info['caution']:
        raise ValueError('Reduce run size before proceeding')
    results[ratio] = simulate(scenario)

fig, ax = plt.subplots(figsize=(8,4))
for ratio, r in results.items():
    ax.plot(r['x_m'], r['bed_m'][-1]-r['bed_m'][0], label=f'feed / capacity = {ratio}')
ax.set(xlabel='Distance downstream (m)',ylabel='Bed elevation change (m)',title='Bed adjustment')
ax.legend(); ax.grid(alpha=.25)
plt.show()
for ratio,r in results.items():
    print(f'{ratio:.1f}: steps={r["steps"]}, storage={r["storage_change_m3"]:.6g} m³, '
          f'mass residual={r["mass_balance_error_m3"]:.2e} m³')


## 3. Interpret

1. Which scenario aggrades, which degrades, and why? Were your predictions correct?
2. Explain why the changes are concentrated near the upstream boundary in these short runs.
3. How close is the sediment mass balance? Does a small residual prove that the simulated bed evolution is physically correct?
4. Change **only one** of grain size, channel width (uniform throughout), discharge, or slope. Predict the result, then rerun the preflight and experiment.
5. Identify two situations in which the normal-flow approximation would be inappropriate.

**Optional convergence check:** Compare final bed changes at $\Delta x=5$ and $2.5$ m, reducing $\Delta t$ appropriately. Distinguish stability from agreement under refinement.